# MILO - Cell type composition analysis

Publication [Dann et al, 2021](https://www.nature.com/articles/s41587-021-01033-z)  
Marioni Lab [GitHub](https://github.com/MarioniLab/miloR)  

Theis lab python adaption with [pertpy](https://github.com/theislab/pertpy)

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # miloR
            library(miloR), 
            library(ggbeeswarm), 
            
            # SingleCellExperiment
            library(SingleCellExperiment), 

            # Data 
            library(tidyverse), 
            library(data.table), 

            # Plotting 
            library(patchwork), 
            library(cowplot), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv="p.3.8.12-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

In [ ]:
# Parallel 
BPPARAM <- SerialParam()
register(BPPARAM)

In [ ]:
# Reticulate python modules
sc <- import("scanpy", as="sc", convert=FALSE)
ad <- import("anndata", as="ad", convert=FALSE)

# Import Seurat object

In [ ]:
# Import Seurat object 
so <- readRDS("data/object/pp_subset.rds") # Important to load the same data as in the milo subsets to compute cell size and samples

# Import Seurat object with SCVI latent space from AnnData

In [ ]:
milo_import <- function(so, h5ad_file, hto_demux_class, domain_leiden_subset) {
    
    # Get Seurat subsets
    so <- subset(so, subset=hto_demux_class=={{hto_demux_class}} & domain_leiden_subset=={{domain_leiden_subset}})
    
    print(dim(so))
    
    # Convert to SCE
    sce <- as.SingleCellExperiment(so)
 
    # Import AnnData
    adata <- sc$read_h5ad(h5ad_file)
    
    # Subset latent space 
    X_scvi <- py_to_r(adata$obsm["X_scvi"])
    
    print(dim(X_scvi))
    
    # Add latent space to SCE object 
    reducedDims(sce) <- list("X_scvi"=X_scvi)
    
    # Create Milo object
    milo <- miloR::Milo(sce)

    return(milo)
    
}

In [ ]:
milo_1 <- milo_import(so, "data/object/subsets/annotation/lymphocyte_blood.h5ad", "Blood", "lymphocyte")
milo_2 <- milo_import(so, "data/object/subsets/annotation/lymphocyte_skin.h5ad", "Skin", "lymphocyte")
milo_3 <- milo_import(so, "data/object/subsets/annotation/myeloid_blood.h5ad", "Blood", "myeloid")
milo_4 <- milo_import(so, "data/object/subsets/annotation/myeloid_skin.h5ad", "Skin", "myeloid")
milo_5 <- milo_import(so, "data/object/subsets/annotation/structural.h5ad", "Skin", "structural")

# Run Milo

In [ ]:
k_range <- seq(10, 100, 10)

In [ ]:
cache <- TRUE

In [ ]:
if(!cache) {
    
    for (k in k_range) {
        
        # Run makeNhoods with refinement_scheme="graph" to avoid calcNhoodDistance and set fdr.weighting="graph-only" testNhoods
        # https://github.com/MarioniLab/miloR/issues/99
    
        # Graph 
        milo_1 <- buildGraph(milo_1, k=k, d=10, reduced.dim="X_scvi")
        milo_2 <- buildGraph(milo_2, k=k, d=10, reduced.dim="X_scvi")
        milo_3 <- buildGraph(milo_3, k=k, d=10, reduced.dim="X_scvi")
        milo_4 <- buildGraph(milo_4, k=k, d=10, reduced.dim="X_scvi")
        milo_5 <- buildGraph(milo_5, k=k, d=10, reduced.dim="X_scvi")

        milo_1 <- makeNhoods(milo_1, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="X_scvi", refinement_scheme="graph")
        milo_2 <- makeNhoods(milo_2, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="X_scvi", refinement_scheme="graph")
        milo_3 <- makeNhoods(milo_3, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="X_scvi", refinement_scheme="graph")
        milo_4 <- makeNhoods(milo_4, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="X_scvi", refinement_scheme="graph")
        milo_5 <- makeNhoods(milo_5, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="X_scvi", refinement_scheme="graph")

        milo_1 <- countCells(milo_1, meta.data=data.frame(colData(milo_1)), sample="sample_name")
        milo_2 <- countCells(milo_2, meta.data=data.frame(colData(milo_2)), sample="sample_name")
        milo_3 <- countCells(milo_3, meta.data=data.frame(colData(milo_3)), sample="sample_name")
        milo_4 <- countCells(milo_4, meta.data=data.frame(colData(milo_4)), sample="sample_name")
        milo_5 <- countCells(milo_5, meta.data=data.frame(colData(milo_5)), sample="sample_name")

        saveRDS(milo_1, paste0("data/object/milo/time_point/milo_k", as.character(k), "_1.rds"))
        saveRDS(milo_2, paste0("data/object/milo/time_point/milo_k", as.character(k), "_2.rds"))
        saveRDS(milo_3, paste0("data/object/milo/time_point/milo_k", as.character(k), "_3.rds"))
        saveRDS(milo_4, paste0("data/object/milo/time_point/milo_k", as.character(k), "_4.rds"))
        saveRDS(milo_5, paste0("data/object/milo/time_point/milo_k", as.character(k), "_5.rds"))
    
    }
    
}

In [ ]:
milo_1 <- lapply(k_range, function(k) {readRDS(paste0("data/object/milo/time_point/milo_k", as.character(k), "_1.rds"))})
milo_2 <- lapply(k_range, function(k) {readRDS(paste0("data/object/milo/time_point/milo_k", as.character(k), "_2.rds"))})
milo_3 <- lapply(k_range, function(k) {readRDS(paste0("data/object/milo/time_point/milo_k", as.character(k), "_3.rds"))})
milo_4 <- lapply(k_range, function(k) {readRDS(paste0("data/object/milo/time_point/milo_k", as.character(k), "_4.rds"))})
milo_5 <- lapply(k_range, function(k) {readRDS(paste0("data/object/milo/time_point/milo_k", as.character(k), "_5.rds"))})

In [ ]:
names(milo_1) <- k_range
names(milo_2) <- k_range
names(milo_3) <- k_range
names(milo_4) <- k_range
names(milo_5) <- k_range

In [ ]:
options(repr.plot.width=25, repr.plot.height=2.5)

hist_1 <- lapply(milo_1, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Blood lymphocyte\n k", i@.k)) + xlim(0, 300) + theme_global_set(2)})
hist_2 <- lapply(milo_2, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Skin lymphocyte\n k", i@.k)) + xlim(0, 300) + theme_global_set(2)})
hist_3 <- lapply(milo_3, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Blood myeloid\n k", i@.k)) + xlim(0, 300) + theme_global_set(2)})
hist_4 <- lapply(milo_4, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Skin myeloid\n k", i@.k)) + xlim(0, 300) + theme_global_set(2)})
hist_5 <- lapply(milo_5, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Skin structural\n k", i@.k)) + xlim(0, 300) + theme_global_set(2)})

wrap_plots(hist_1, ncol=10, nrow=1) 
wrap_plots(hist_2, ncol=10, nrow=1) 
wrap_plots(hist_3, ncol=10, nrow=1)
wrap_plots(hist_4, ncol=10, nrow=1)
wrap_plots(hist_5, ncol=10, nrow=1)

In [ ]:
testNhoods <- function(milo, counts, design, model_contrasts, cell_sizes, fdr_weighting="graph-overlap", reduced.dim="X_scvi") {
    
    dge <- DGEList(counts=counts, lib.size=cell_sizes)
    dge <- calcNormFactors(dge, method="TMM")
    dge <- estimateDisp(dge, design)
    
    fit <- glmQLFit(dge, design, robust=TRUE)
    
    model_contrasts <- makeContrasts(contrasts=model_contrasts, levels=design)
    res <- as.data.frame(topTags(glmQLFTest(fit, contrast=model_contrasts), sort.by='none', n=Inf))
    
    rownames(res) <- as.numeric(1:nrow(counts))
    res <- res[1:nrow(nhoodCounts(milo)), ]
    
    res$Nhood <- as.numeric(rownames(nhoodCounts(milo)))
    
    mod.spatialfdr <- graphSpatialFDR(

        x.nhoods=nhoods(milo),
        graph=graph(milo),
        weighting=fdr_weighting,
        k=milo@.k,
        pvalues=res[order(res$Nhood), ]$PValue,
        indices=nhoodIndex(milo),
        distances=nhoodDistances(milo),
        reduced.dimensions=reducedDim(milo, reduced.dim)
    
    )
    
    res$SpatialFDR[order(res$Nhood)] <- mod.spatialfdr
    
    return(res)
    
}

In [ ]:
milo_glm <- function(milo, counts, meta, hto_demux_class, domain_leiden_subset) {
    
    # Subset meta data 
    meta <- meta[meta[["hto_demux_class"]]==hto_demux_class & meta[["domain_leiden_subset"]] %in% domain_leiden_subset, ]
    
    # Compute cell sizes 
    cell_sizes <- setNames(table(meta$sample_name), names(table(meta$sample_name)))

    # Design 
    design_df <- meta[,c ("sample_name", "time_point", "patient_id"), drop=FALSE]
    design_df <- distinct(design_df)
    rownames(design_df) <- design_df$sample_name
    
    # Test data 
    sample_keep <- colnames(counts)[colSums(counts)>0]
    design_df <- design_df[sample_keep, ]
    cell_sizes <- cell_sizes[sample_keep]
    counts <- counts[, sample_keep]
    
    # Model matrix
    design <- model.matrix(~0+time_point, data=design_df)
    
    result_1 <- testNhoods(milo, counts, design, model_contrasts="time_pointTx-time_pointBaseline", cell_sizes)
    result_2 <- testNhoods(milo, counts, design, model_contrasts="time_pointD14-time_pointBaseline", cell_sizes)
    result_3 <- testNhoods(milo, counts, design, model_contrasts="time_pointD100-time_pointBaseline", cell_sizes)
    result_4 <- testNhoods(milo, counts, design, model_contrasts="time_pointGVHD-time_pointBaseline", cell_sizes)
    
    # P-value hist
    pvalue_hist_1 <- ggplot(result_1, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs Tx\n k", milo@.k)) + theme_global_set(2)
    pvalue_hist_2 <- ggplot(result_2, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs D14\n k", milo@.k)) + theme_global_set(2)
    pvalue_hist_3 <- ggplot(result_3, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs D100\n k", milo@.k)) + theme_global_set(2)
    pvalue_hist_4 <- ggplot(result_4, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs GVHD\n k", milo@.k)) + theme_global_set(2)

    # Volcano plot 
    vp_1 <- ggplot(result_1, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs Tx\n k", milo@.k)) + theme_global_set(2)
    vp_2 <- ggplot(result_2, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs D14\n k", milo@.k)) + theme_global_set(2)
    vp_3 <- ggplot(result_3, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs D100\n k", milo@.k)) + theme_global_set(2)
    vp_4 <- ggplot(result_4, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs GVHD\n k", milo@.k)) + theme_global_set(2)
    
    # Annotate hoods
    result_1 <- annotateNhoods(milo, result_1, coldata_col="cell_type_leiden_subset")
    result_2 <- annotateNhoods(milo, result_2, coldata_col="cell_type_leiden_subset")
    result_3 <- annotateNhoods(milo, result_3, coldata_col="cell_type_leiden_subset")
    result_4 <- annotateNhoods(milo, result_4, coldata_col="cell_type_leiden_subset")
    
    # Factor names 
    result_1$cell_type_leiden_subset <- factor(result_1$cell_type_leiden_subset, levels=rev(names(color$cell_type_leiden_subset_r))) %>% droplevels()
    result_2$cell_type_leiden_subset <- factor(result_2$cell_type_leiden_subset, levels=rev(names(color$cell_type_leiden_subset_r))) %>% droplevels()
    result_3$cell_type_leiden_subset <- factor(result_3$cell_type_leiden_subset, levels=rev(names(color$cell_type_leiden_subset_r))) %>% droplevels()
    result_4$cell_type_leiden_subset <- factor(result_4$cell_type_leiden_subset, levels=rev(names(color$cell_type_leiden_subset_r))) %>% droplevels()
    
    # Beeswarm 
    dabp_1 <- suppressMessages(plotDAbeeswarm(result_1, group.by="cell_type_leiden_subset", alpha=0.1) + ylab("") + ggtitle(paste("Baseline vs Tx\n k", milo@.k)) + theme_global_set(2))
    dabp_2 <- suppressMessages(plotDAbeeswarm(result_2, group.by="cell_type_leiden_subset", alpha=0.1) + ylab("") + ggtitle(paste("Baseline vs D14\n k", milo@.k))  + theme_global_set(2))
    dabp_3 <- suppressMessages(plotDAbeeswarm(result_3, group.by="cell_type_leiden_subset", alpha=0.1) + ylab("") + ggtitle(paste("Baseline vs D100\n k", milo@.k))  + theme_global_set(2))
    dabp_4 <- suppressMessages(plotDAbeeswarm(result_4, group.by="cell_type_leiden_subset", alpha=0.1) + ylab("") + ggtitle(paste("Baseline vs GVHD\n k", milo@.k)) + theme_global_set(2))

    # Combine plots 
    res_p <- pvalue_hist_1 + pvalue_hist_2 + pvalue_hist_3 + pvalue_hist_4 + vp_1 + vp_2 + vp_3 + vp_4 + dabp_1 + dabp_2 + dabp_3 + dabp_4 + plot_layout(ncol=12)
    
    # Cobine result table
    res_da <- list(result_1, result_2, result_3, result_4)
    
    # Combine all
    resl <- list(res_p=res_p, res_da=res_da)
    
    return(resl)
    
}

## Blood lymphoid 

In [ ]:
counts <- lapply(seq_along(k_range), function(k){
    
    counts_1 <- nhoodCounts(milo_1[[k]]) %>% as.data.frame()
    counts_2 <- nhoodCounts(milo_3[[k]]) %>% as.data.frame()
    counts <- dplyr::bind_rows(counts_1, counts_2)
    counts[is.na(counts)] <- 0
    
    return(counts)
    
}
                )

In [ ]:
resl_1 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_1[[k]], counts=counts[[k]], meta=so@meta.data, hto_demux_class="Blood", domain_leiden_subset=c("lymphocyte", "myeloid")))

In [ ]:
# saveRDS(sapply(resl_1, '[', 2), "data/object/milo/time_point/res_1.rds")

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_1, "[", 1), ncol=1, nrow=10)

## Skin lymphoid

In [ ]:
counts <- lapply(seq_along(k_range), function(k){
    
    counts_1 <- nhoodCounts(milo_2[[k]]) %>% as.data.frame()
    counts_2 <- nhoodCounts(milo_4[[k]]) %>% as.data.frame()
    counts <- dplyr::bind_rows(counts_1, counts_2)
    counts[is.na(counts)] <- 0
    
    return(counts)
    
}
                )

In [ ]:
resl_2 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_2[[k]], counts=counts[[k]], meta=so@meta.data, hto_demux_class="Skin", domain_leiden_subset=c("lymphocyte", "myeloid")))

In [ ]:
# saveRDS(sapply(resl_2, '[', 2), "data/object/milo/time_point/res_2.rds")

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_2, "[", 1), ncol=1, nrow=10)

## Blood myeloid

In [ ]:
counts <- lapply(seq_along(k_range), function(k){
    
    counts_1 <- nhoodCounts(milo_3[[k]]) %>% as.data.frame()
    counts_2 <- nhoodCounts(milo_1[[k]]) %>% as.data.frame()
    counts <- dplyr::bind_rows(counts_1, counts_2)
    counts[is.na(counts)] <- 0
    
    return(counts)
    
}
                )

In [ ]:
resl_3 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_3[[k]], counts=counts[[k]], meta=so@meta.data, hto_demux_class="Blood", domain_leiden_subset=c("lymphocyte", "myeloid")))

In [ ]:
# saveRDS(sapply(resl_3, '[', 2), "data/object/milo/time_point/res_3.rds")

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_3, "[", 1), ncol=1, nrow=10)

## Skin myeloid

In [ ]:
counts <- lapply(seq_along(k_range), function(k){
    
    counts_1 <- nhoodCounts(milo_4[[k]]) %>% as.data.frame()
    counts_2 <- nhoodCounts(milo_2[[k]]) %>% as.data.frame()
    counts <- dplyr::bind_rows(counts_1, counts_2)
    counts[is.na(counts)] <- 0
    
    return(counts)
    
}
                )

In [ ]:
resl_4 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_4[[k]], counts=counts[[k]], meta=so@meta.data, hto_demux_class="Skin", domain_leiden_subset=c("lymphocyte", "myeloid")))

In [ ]:
# saveRDS(sapply(resl_4, '[', 2), "data/object/milo/time_point/res_4.rds")

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_4, "[", 1), ncol=1, nrow=10)

## Skin structural

In [ ]:
counts <- lapply(seq_along(k_range), function(k){
    
    counts_1 <- nhoodCounts(milo_5[[k]]) %>% as.data.frame()
    
    return(counts_1)
    
}
                )

In [ ]:
resl_5 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_5[[k]], counts=counts[[k]], meta=so@meta.data, hto_demux_class="Skin", domain_leiden_subset=c("structural")))

In [ ]:
# saveRDS(sapply(resl_5, '[', 2), "data/object/milo/time_point/res_5.rds")

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_5, "[", 1), ncol=1, nrow=10)

# Session info

In [ ]:
sessionInfo()